# Facial Recognition Using Eigenfaces and Fisherfaces
## Mathematical Foundations, Complete Working, and Interpretation

**A markdown-only study notebook**

This notebook explains the mathematics and the complete processing pipeline described in the project report *Facial Recognition Using Eigenfaces and Fisherfaces*. It is intentionally made entirely of Markdown cells: there are no executable code cells.

The goal is to understand not only what each algorithm does, but also why each matrix is constructed, what every dimension means, how a new face is represented, and how the final identity is selected.

> **Central idea:** A face image begins as 4,800 pixel values. The system learns a smaller coordinate system from the training faces, represents every face in that coordinate system, and compares a test face with the stored training faces using cosine similarity.

## Contents

1. Problem definition and experiment design
2. Notation and matrix dimensions
3. From an image to a numerical vector
4. Training and test matrices
5. Mean face and centering
6. PCA: the mathematical objective
7. PCA through the Gram matrix
8. Converting sample-space eigenvectors into eigenfaces
9. Explained variance and component selection
10. The exact Eigenfaces representation used
11. LDA: the mathematical objective
12. Within-class and between-class scatter
13. Fisher's criterion and the generalized eigenvalue problem
14. Why PCA comes before LDA
15. Ridge regularization
16. Constructing and encoding Fisherfaces
17. Feature normalization and cosine similarity
18. Gallery matching and identity-level scores
19. Rank-1 and Top-5 evaluation
20. Results and error analysis
21. Complete end-to-end system
22. Limitations, implementation details, and key numbers
23. Glossary and revision sheet

## 1. Problem definition and experiment design

The task is **closed-set face recognition**. A test image is assumed to belong to one of the people already represented in the training gallery. The system must choose one of those known identities.

The report uses 38 identities:

$$
s_1,s_2,\ldots,s_{38}.
$$

Each selected identity has ten images. Images 1–8 are used for training, while images 9–10 are reserved for testing.

| Role | Images per identity | Total |
|---|---:|---:|
| Training / gallery | 8 | $38\times 8=304$ |
| Testing / queries | 2 | $38\times 2=76$ |
| Total | 10 | $38\times 10=380$ |

Folders $s_{39}$ and $s_{40}$ are excluded. The fixed split makes the comparison repeatable and ensures that Eigenfaces and Fisherfaces use the same examples.

**Important:** The system does not have an unknown-person class or a rejection threshold. Even an unfamiliar person's image would be assigned to the best-scoring one of the 38 known identities.

## 2. Notation and matrix dimensions

Every processed image is resized to 60 pixels wide and 80 pixels high. Therefore, one image contains

$$
d=60\times80=4800
$$

pixel positions.

The report stores images as **columns** of the data matrix.

| Symbol | Meaning | Shape |
|---|---|---:|
| $d$ | Number of pixel features | $4800$ |
| $n$ | Number of training images | $304$ |
| $q$ | Number of test images | $76$ |
| $X_G$ | Training/gallery matrix | $4800\times304$ |
| $X_K$ | Test/query matrix | $4800\times76$ |
| $\mu$ | Mean training face | $4800\times1$ |
| $X_c$ | Centered training matrix | $4800\times304$ |

A column of $X_G$ is one complete face vector. A row corresponds to one pixel position across all training images.

The letter $c$ will denote an identity/class. The notation $n_c$ denotes the number of training images in class $c$; here $n_c=8$ for every class.

## 3. From a photograph to a numerical vector

A photograph must be converted into a fixed-length numerical representation before matrix operations can be applied.

### 3.1 Open, orient, and convert to grayscale

The image is opened, EXIF orientation metadata is applied, and the image is converted to grayscale.

A colour image has red, green, and blue channels. A grayscale image has one brightness value per pixel. The experiment uses grayscale because a single intensity channel is sufficient for the chosen classical methods.

### 3.2 Resize every image

Every image is resized to

$$
\text{width}=60,\qquad \text{height}=80.
$$

The resized image therefore has $80\times60=4800$ entries. The same dimensions are essential: vectors of different lengths could not be stacked into one data matrix.

Lanczos resampling is used for resizing.

### 3.3 Histogram equalization

Histogram equalization redistributes grayscale intensities so that the available brightness range is used more effectively.

It can improve global contrast when an image uses only a narrow range of grayscale values. It is not face detection, landmark alignment, or a guarantee that all lighting differences disappear.

### 3.4 Flatten and scale

View the grayscale image as an $80\times60$ matrix and flatten it into a vector:

$$
x=(x_1,x_2,\ldots,x_{4800})^T.
$$

An 8-bit grayscale intensity is between 0 and 255. Dividing by 255 scales the values approximately into $[0,1]$:

$$
x_j^{(0)}=\frac{\text{pixel intensity}_j}{255}.
$$

For a tiny illustration, the image

$$
\begin{bmatrix}
10&20&30\\
40&50&60
\end{bmatrix}
$$

becomes the six-entry vector

$$
(10,20,30,40,50,60)^T.
$$

The actual face image produces 4,800 entries.

## 4. Standardize each image individually

After flattening and scaling, the notebook standardizes each image using that image's own pixel mean and standard deviation.

For one image vector $x\in\mathbb R^{4800}$, define

$$
\bar{x}=\frac{1}{4800}\sum_{j=1}^{4800}x_j
$$

and

$$
s_x=\sqrt{\frac{1}{4800}\sum_{j=1}^{4800}(x_j-\bar{x})^2}.
$$

The standardized pixel vector is

$$
\widetilde{x}_j=\frac{x_j-\bar{x}}{s_x+10^{-6}}.
$$

The small $10^{-6}$ term protects against division by zero or an extremely small standard deviation.

### Why do this?

- Subtracting the image's own mean reduces a global brightness offset.
- Dividing by its own standard deviation reduces a simple contrast-scale difference.
- It makes the average pixel value close to zero and the pixel standard deviation close to one.

This does **not** remove every lighting effect, shadow, pose change, or expression difference.

> There are two different normalizations in this project. This is **per-image pixel standardization**, performed before PCA/LDA. Later, the projected feature vector receives **L2 normalization**, which is used to make dot products equal cosine similarity.

## 5. Constructing the training and test matrices

Each processed image becomes one 4,800-dimensional column.

The training matrix is

$$
X_G=
\begin{bmatrix}
|&|&&|\\
x_1&x_2&\cdots&x_{304}\\
|&|&&|
\end{bmatrix}
\in\mathbb R^{4800\times304}.
$$

The test matrix is

$$
X_K\in\mathbb R^{4800\times76}.
$$

The same preprocessing is applied to both sets.

### Why is the split important?

The training images determine the mean face, PCA directions, LDA directions, and gallery features. The test images are only transformed using the already learned quantities and then evaluated.

The test images must not be used to learn the projection directions; otherwise, the evaluation would no longer be a clean test of the learned representation.

The split is fixed, not random, and the report does not use cross-validation. Thus the reported accuracy describes this particular split.

## 6. Mean face and centering

PCA studies variation around the mean. The mean face is calculated using the training images only.

For pixel position $j$, the mean value across the 304 training faces is

$$
\mu_j=\frac{1}{304}\sum_{i=1}^{304}(X_G)_{ji}.
$$

In matrix notation,

$$
\mu=\frac{1}{304}X_G\mathbf 1_{304}
\in\mathbb R^{4800\times1},
$$

where $\mathbf 1_{304}$ is a column vector of 304 ones.

Subtract the mean face from every training image:

$$
X_c=X_G-\mu\mathbf 1_{304}^{T}.
$$

Hence,

$$
X_c\in\mathbb R^{4800\times304}.
$$

Every column of $X_c$ is the difference between one training face and the average training face.

The test images are centered using this **same training mean**:

$$
X_{K,c}=X_K-\mu\mathbf 1_{76}^{T}.
$$

Do not calculate a new mean from the test set or from one query image. The learned coordinate system must be used consistently.

## 7. PCA: what is the mathematical objective?

**Principal Component Analysis (PCA)** finds directions along which the data varies the most.

Imagine a cloud of points in two dimensions. If the points form a long, narrow cloud, the direction along the long axis captures most of the variation. PCA finds that direction first, then finds a perpendicular direction for the remaining variation.

A face is a point in 4,800-dimensional space. PCA performs the same idea in this high-dimensional space.

### 7.1 Projecting data onto a direction

Let $u\in\mathbb R^{4800}$ be a direction. For centered training image $x_i-\mu$, its coordinate along $u$ is

$$
z_i=u^T(x_i-\mu).
$$

If $u$ is a unit vector, the total squared projected variation is

$$
V(u)=\sum_{i=1}^{304}z_i^2.
$$

Writing all observations as columns gives

$$
V(u)=u^TX_cX_c^Tu.
$$

Define the scatter/covariance-proportional matrix

$$
C=X_cX_c^T.
$$

PCA seeks

$$
\boxed{
\max_{u}\;u^TCu
\quad\text{subject to}\quad u^Tu=1.
}
$$

The unit-length constraint is necessary. Without it, multiplying $u$ by an arbitrarily large constant would make the objective arbitrarily large.

### 7.2 Why eigenvectors appear

Using a Lagrange multiplier $\lambda$,

$$
L(u,\lambda)=u^TCu-\lambda(u^Tu-1).
$$

Differentiating with respect to $u$ and setting the derivative to zero,

$$
2Cu-2\lambda u=0.
$$

Therefore,

$$
Cu=\lambda u.
$$

So the maximizing directions are eigenvectors of $C$. The eigenvector with the largest eigenvalue is the first principal component; the next largest eigenvalue gives the next principal direction, and so on.

The eigenvalue measures the amount of variation associated with its eigenvector.

## 8. PCA through the Gram matrix

A direct PCA calculation in pixel space would diagonalize

$$
X_cX_c^T\in\mathbb R^{4800\times4800}.
$$

But there are only 304 training images. The report uses a smaller matrix:

$$
\boxed{G=X_c^TX_c\in\mathbb R^{304\times304}.}
$$

This is called the **Gram matrix** or sample-space matrix.

It is much smaller to diagonalize: $304\times304$, rather than $4800\times4800$.

### 8.1 Solve the smaller eigenvalue problem

Suppose

$$
Gv_i=\lambda_i v_i,
$$

where $v_i\in\mathbb R^{304}$ and $\lambda_i>0$.

The vector $v_i$ is an eigenvector in sample space. It is not yet an eigenface, because an eigenface must have 4,800 pixel coordinates.

### 8.2 Map it into pixel space

Define

$$
\boxed{
u_i=\frac{X_cv_i}{\sqrt{\lambda_i}}.
}
$$

Then $u_i\in\mathbb R^{4800}$.

To see why it is an eigenvector of the pixel-space matrix:

$$
\begin{aligned}
X_cX_c^Tu_i
&=X_cX_c^T\frac{X_cv_i}{\sqrt{\lambda_i}}\\
&=\frac{X_c(X_c^TX_c)v_i}{\sqrt{\lambda_i}}\\
&=\frac{X_cGv_i}{\sqrt{\lambda_i}}\\
&=\frac{X_c(\lambda_i v_i)}{\sqrt{\lambda_i}}\\
&=\lambda_i u_i.
\end{aligned}
$$

Thus $u_i$ is a pixel-space eigenvector with eigenvalue $\lambda_i$.

The notebook removes eigenvalues that are numerically zero before dividing by $\sqrt{\lambda_i}$. This avoids unstable division.

### 8.3 What is an eigenface?

An eigenface is a PCA direction represented in the original pixel coordinate system. Reshape its 4,800 entries into an $80\times60$ image to visualize it.

An eigenface is **not** a photograph of a separate person. It is a learned pattern of positive and negative pixel variation around the mean face.

## 9. Explained variance and component selection

Let the positive PCA eigenvalues, in descending order, be

$$
\lambda_1\geq\lambda_2\geq\cdots\geq\lambda_r>0.
$$

The cumulative explained-variance fraction of the first $k$ components is

$$
\boxed{
C_k=\frac{\lambda_1+\lambda_2+\cdots+\lambda_k}
{\lambda_1+\lambda_2+\cdots+\lambda_r}.
}
$$

For example, $C_{100}=0.90$ means that the first 100 components account for 90% of the total eigenvalue mass (and hence 90% of the total centered variation represented by these components).

The Eigenfaces settings are:

| Parameter | Value | Meaning |
|---|---:|---|
| `EIGEN_VAR` | 0.95 | Target cumulative explained variance |
| `EIGEN_MAX` | 300 | Maximum selected boundary |
| `EIGEN_DROP_FIRST` | 3 | Number of leading components later omitted |

The selection procedure is:

1. Find the first $k$ for which $C_k\geq0.95$.
2. Do not allow $k$ to be below 50.
3. Do not allow $k$ to exceed 300 or the number of available eigenfaces.

For this saved run,

$$
K_{\text{EIG}}=158,
\qquad C_{158}=95.06\%.
$$

This 95.06% is the cumulative variance of the first 158 components before the first three are dropped.

## 10. The exact Eigenfaces representation used

The implementation discards the first three selected PCA directions.

In zero-based Python slicing, the basis is selected using columns 3 through 157:

$$
U_{\text{EIG}}= [u_4,u_5,\ldots,u_{158}].
$$

Therefore, the number of retained directions is

$$
158-3=\boxed{155}.
$$

The first-three-component removal is a heuristic setting of this implementation. The report does not provide a separate proof that the first three components are always unhelpful.

### 10.1 Encode one face

For a preprocessed face vector $x$, first center it using the training mean and then project it:

$$
\boxed{
z_E=U_{\text{EIG}}^T(x-\mu)\in\mathbb R^{155}.
}
$$

The original face has 4,800 values. Its Eigenfaces representation has only 155 coordinates.

The implementation then L2-normalizes this feature vector; that operation is explained later.

### 10.2 Batch dimensions

| Quantity | Shape |
|---|---:|
| $U_{\text{EIG}}$ | $4800\times155$ |
| Training gallery features | $304\times155$ |
| Test query features | $76\times155$ |

The transpose is used so that each image becomes one row in the feature matrix, making row-wise normalization convenient.

## 11. LDA: what is the mathematical objective?

**Linear Discriminant Analysis (LDA)** uses the identity labels. Its goal is to find directions that:

- keep examples of the same identity close together;
- make examples from different identities far apart.

PCA asks:

> In which directions do the training faces vary most overall?

LDA asks:

> In which directions are the known identities best separated?

This difference is central. A direction with large overall variation is not necessarily useful for distinguishing people. For example, a large brightness change may contribute strongly to PCA variance without separating identities.

LDA uses the class labels to focus on identity separation.

## 12. PCA reduction before LDA

The original dimension is 4,800, while there are only 304 training images. Direct LDA in the original pixel space can have a singular within-class scatter matrix.

The report first projects the centered training data onto the first 300 PCA directions.

Let

$$
P=[u_1,u_2,\ldots,u_{300}]
\in\mathbb R^{4800\times300}.
$$

The PCA-reduced coordinates are

$$
Y=(P^TX_c)^T\in\mathbb R^{304\times300}.
$$

Each row of $Y$ is one training image represented by 300 PCA coordinates.

**Important implementation detail:** Fisherfaces uses the first 300 PCA directions from `evecs`. It does not use the Eigenfaces basis after the first-three-component drop.

## 13. Class means and the overall mean in PCA space

There are $C=38$ identities, each with $n_c=8$ training observations.

Let $y_i\in\mathbb R^{300}$ be the PCA representation of training image $i$, and let $c(i)$ denote its identity.

For class $c$, its mean vector is

$$
\boxed{
\mu_c=\frac{1}{n_c}\sum_{i:c(i)=c}y_i
\in\mathbb R^{300}.
}
$$

The overall mean in PCA space is

$$
\boxed{
\mu=\frac{1}{304}\sum_{i=1}^{304}y_i
=\frac{1}{304}\sum_{c=1}^{38}n_c\mu_c.
}
$$

The class mean describes the average representation of one identity. The overall mean describes the average representation across all training identities.

## 14. Within-class scatter $S_W$

Within-class scatter measures how much images belonging to the **same identity** vary around that identity's own mean.

For image $i$, define its within-class residual:

$$
r_i=y_i-\mu_{c(i)}.
$$

The within-class scatter matrix is

$$
\boxed{
S_W=\sum_{i=1}^{304}r_ir_i^T
=\sum_{c=1}^{38}\sum_{i:c(i)=c}
(y_i-\mu_c)(y_i-\mu_c)^T.
}
$$

Its dimension is

$$
S_W\in\mathbb R^{300\times300}.
$$

### Interpretation

Suppose one person's eight training images differ because of expression, pose, or lighting. Those differences contribute to $S_W$.

A good LDA direction should have **small projected within-class variation**.

For a direction $v$, the projected within-class scatter is

$$
v^TS_Wv.
$$

## 15. Between-class scatter $S_B$

Between-class scatter measures how far the identity means lie from the overall mean.

For identity $c$, its mean offset is

$$
d_c=\mu_c-\mu.
$$

The between-class scatter matrix is

$$
\boxed{
S_B=\sum_{c=1}^{38}n_c(\mu_c-\mu)(\mu_c-\mu)^T.
}
$$

Its dimension is also

$$
S_B\in\mathbb R^{300\times300}.
$$

The factor $n_c$ weights each class by its number of training observations. In this experiment, all classes have eight observations.

For a direction $v$, the projected between-class scatter is

$$
v^TS_Bv.
$$

A good LDA direction should have **large projected between-class separation**.

| Matrix | What it measures | LDA wants |
|---|---|---|
| $S_W$ | Spread within the same identity | Small |
| $S_B$ | Separation among identity means | Large |

## 16. Fisher's discriminant criterion

LDA combines the two goals into one ratio:

$$
\boxed{
J(v)=\frac{v^TS_Bv}{v^TS_Wv}.
}
$$

The numerator becomes large when class means are well separated along $v$. The denominator becomes small when images of the same identity are tightly grouped along $v$.

Therefore, maximizing $J(v)$ seeks a direction with strong identity separation relative to within-identity variation.

As in PCA, the scale of $v$ must be controlled. One can impose a normalization constraint such as

$$
v^TS_Wv=1
$$

when the denominator is positive, or formulate the equivalent generalized eigenvalue problem.

The Fisher criterion is not an accuracy percentage. It is an objective value associated with a discriminant direction.

## 17. The generalized eigenvalue problem

Maximizing the Fisher criterion leads to the generalized eigenvalue equation

$$
\boxed{
S_Bv=\lambda S_Wv.
}
$$

Equivalently, when $S_W$ is invertible,

$$
S_W^{-1}S_Bv=\lambda v.
$$

Thus the discriminant directions are generalized eigenvectors, and the associated eigenvalues indicate the Fisher criterion achieved by those directions.

The implementation uses a symmetric generalized eigensolver on the pair $(S_B,S_W)$, then sorts the eigenvalues in descending order.

The directions with the largest eigenvalues are retained because they provide the strongest between-class separation relative to within-class scatter under this criterion.

## 18. Ridge regularization of within-class scatter

The within-class scatter matrix may be singular or numerically unstable. The notebook therefore adds a small positive diagonal term.

The configuration is

$$
\text{FISHER\_RIDGE}=10^{-2},\qquad p=300.
$$

First calculate

$$
\gamma=10^{-2}\frac{\operatorname{tr}(S_W)}{300},
$$

where $\operatorname{tr}(S_W)$ is the sum of the diagonal entries.

Then replace the within-class scatter by

$$
\boxed{
S_W^{\text{reg}}=S_W+\gamma I_{300}.
}
$$

The generalized eigenproblem is therefore

$$
\boxed{
S_Bv=\lambda S_W^{\text{reg}}v.
}
$$

The identity matrix has ones on its diagonal and zeros elsewhere. Adding $\gamma I$ stabilizes the calculation and helps avoid problems caused by very small or zero eigenvalues.

Regularization does not add face images. It changes the matrix calculation.

## 19. Why are there only 37 Fisherfaces?

A fundamental property of LDA is that the number of nonzero independent discriminant directions is at most

$$
C-1,
$$

where $C$ is the number of classes.

Here,

$$
C=38
\quad\Longrightarrow\quad
C-1=37.
$$

The implementation sets the number of directions to

$$
N_F=\min(C-1,\ p,\ \text{FISHER\_DIM}).
$$

With $p=300$ and `FISHER_DIM = 200`,

$$
N_F=\min(37,300,200)=\boxed{37}.
$$

So `FISHER_DIM = 200` is only an upper cap. It does not mean that 200 Fisherfaces are produced.

## 20. Mapping LDA directions back to pixel space

The LDA eigenvectors are directions in the 300-dimensional PCA coordinate system. To project a 4,800-dimensional face directly, the directions are mapped back through the PCA matrix.

Let $V_{37}\in\mathbb R^{300\times37}$ contain the 37 selected LDA eigenvectors. Define

$$
\boxed{
W_{\text{FISH}}=PV_{37}
\in\mathbb R^{4800\times37}.
}
$$

Each column of $W_{\text{FISH}}$ is a Fisherface direction in pixel space.

For a preprocessed face $x$, its Fisherface representation is

$$
\boxed{
z_F=W_{\text{FISH}}^T(x-\mu)\in\mathbb R^{37}.
}
$$

The feature is then L2-normalized.

| Quantity | Shape |
|---|---:|
| PCA matrix $P$ | $4800\times300$ |
| LDA directions $V_{37}$ | $300\times37$ |
| Fisherface matrix $W_{\text{FISH}}$ | $4800\times37$ |
| Training gallery features | $304\times37$ |
| Test query features | $76\times37$ |

## 21. L2 normalization of the learned feature vectors

After PCA or Fisherface projection, an image is represented by a shorter feature vector.

Let

$$
z=(z_1,z_2,\ldots,z_k)^T.
$$

Its Euclidean length, also called its L2 norm, is

$$
\|z\|_2=\sqrt{z_1^2+z_2^2+\cdots+z_k^2}.
$$

The normalized vector is

$$
\boxed{
\widehat z=\frac{z}{\max(\|z\|_2,10^{-8})}.
}
$$

For a nonzero vector with norm at least $10^{-8}$, this gives

$$
\|\widehat z\|_2=1.
$$

The small floor prevents division by zero. A zero vector remains zero.

### Why normalize the projected features?

Consider

$$
a=(1,1),\qquad b=(10,10).
$$

They point in exactly the same direction, but their lengths differ. A raw dot product is affected by their magnitudes. After L2 normalization, both point in the same unit direction, so their dot product is 1.

This makes the later comparison focus on the direction/pattern of the learned representation rather than its overall length.

## 22. Cosine similarity

For two nonzero vectors $a$ and $b$, cosine similarity is

$$
\boxed{
\operatorname{cosSim}(a,b)=
\frac{a^Tb}{\|a\|_2\|b\|_2}.
}
$$

It is the cosine of the angle between the vectors.

| Cosine similarity | Geometric meaning |
|---:|---|
| $1$ | Same direction; angle $0^\circ$ |
| $0$ | Perpendicular; angle $90^\circ$ |
| $-1$ | Opposite directions; angle $180^\circ$ |

For nonzero vectors, cosine similarity lies between $-1$ and $1$.

### Why does the dot product become cosine similarity?

If both vectors are L2-normalized, then

$$
\|\widehat a\|_2=\|\widehat b\|_2=1.
$$

Therefore,

$$
\operatorname{cosSim}(\widehat a,\widehat b)
=\frac{\widehat a^T\widehat b}{1\cdot1}
=\boxed{\widehat a^T\widehat b}.
$$

So the notebook can calculate cosine similarity using a simple dot product.

### Similarity is not distance

A common cosine-distance definition is

$$
d_{\cos}(a,b)=1-\operatorname{cosSim}(a,b).
$$

The notebook does **not** calculate this distance. It calculates cosine similarity and sorts from largest to smallest. Calling its score a cosine distance would be incorrect.

For unit vectors, Euclidean distance and cosine similarity give the same ranking because

$$
\|\widehat a-\widehat b\|_2^2
=2-2\operatorname{cosSim}(\widehat a,\widehat b).
$$

## 23. The gallery: stored reference faces

The **gallery** is the collection of 304 training images and their learned feature vectors.

Each gallery feature vector is stored with:
- its identity label (such as $s_1$);
- its original image information, which is useful for displaying diagnostic matches.

There are eight gallery photographs for every identity.

The **query** is one of the 76 test images. Its identity is not used to compute its similarity scores. The query is simply transformed using the already learned mean and projection directions.

The Eigenfaces gallery and Fisherfaces gallery are different representations of the same original training images:
- Eigenfaces gallery: $304\times155$;
- Fisherfaces gallery: $304\times37$.

A query must be encoded using the corresponding method. Comparing an Eigenfaces query to a Fisherfaces gallery is invalid because their feature dimensions and coordinate systems differ.

## 24. Pairwise query-to-gallery similarity matrix

Let $Q$ be the matrix of normalized query features and $G$ the matrix of normalized gallery features.

For either method, the notebook computes

$$
\boxed{S_{\text{image}}=QG^T.}
$$

For Eigenfaces,

$$
Q\in\mathbb R^{76\times155},
\qquad
G\in\mathbb R^{304\times155}.
$$

For Fisherfaces,

$$
Q\in\mathbb R^{76\times37},
\qquad
G\in\mathbb R^{304\times37}.
$$

In both cases,

$$
S_{\text{image}}\in\mathbb R^{76\times304}.
$$

The entry $S_{\text{image}}[i,j]$ is the cosine similarity between test query $i$ and gallery photograph $j$.

There are

$$
76\times304=\boxed{23,104}
$$

query-to-gallery comparisons per method.

The matrix multiplication calculates all these dot products together.

## 25. Convert image-level scores into identity-level scores

There are 304 gallery images but only 38 identities. Each identity has eight stored photographs.

For a query feature vector $q$, let

$$
g_{c,1},g_{c,2},\ldots,g_{c,8}
$$

be the normalized gallery feature vectors for identity $c$.

The notebook assigns identity $c$ the maximum similarity among its eight photographs:

$$
\boxed{
S_c(q)=\max_{r\in\{1,\ldots,8\}}q^Tg_{c,r}.
}
$$

This is a **maximum-over-templates** rule.

It does not average the eight scores and does not create one class-centroid vector. It allows any one of the eight stored photographs to provide the strongest match.

### Small illustrative example

Suppose a query has these best image-level similarities:

| Identity | Eight similarities (illustrative) | Identity score |
|---|---|---:|
| $s_1$ | 0.42, 0.71, 0.55, 0.38, 0.49, 0.62, 0.44, 0.51 | 0.71 |
| $s_2$ | 0.63, 0.88, 0.60, 0.52, 0.74, 0.69, 0.57, 0.66 | 0.88 |
| $s_3$ | 0.79, 0.51, 0.58, 0.61, 0.46, 0.70, 0.64, 0.55 | 0.79 |

The identity ranking is $s_2$, then $s_3$, then $s_1$.

**Trade-off:** The maximum rule can match a query to the most suitable pose/expression among the templates, but it can also be sensitive to one accidental high similarity.

## 26. Rank the 38 identities

After calculating one score for every identity, sort the identities in descending order of score.

The predicted identity is

$$
\boxed{
\widehat c(q)=\arg\max_{c\in\{s_1,\ldots,s_{38}\}}S_c(q).
}
$$

- **Rank 1** is the identity with the largest score.
- **Rank 2** is the identity with the second-largest score.
- **Top-5** is the list of the five identities with the largest scores.

Top-5 is calculated after grouping gallery images by identity. It is not the five highest-scoring photographs out of all 304 gallery photographs.

## 27. Evaluation: Rank-1 accuracy

The test set has 76 images with known ground-truth identities.

Let $c_i$ be the true identity of query $i$, and $\widehat c_i$ its Rank-1 prediction.

Define the indicator

$$
\mathbf 1\{\widehat c_i=c_i\}=
\begin{cases}
1,&\widehat c_i=c_i,\\
0,&\widehat c_i\ne c_i.
\end{cases}
$$

Then

$$
\boxed{
\operatorname{Rank1}
=\frac{1}{76}\sum_{i=1}^{76}
\mathbf 1\{\widehat c_i=c_i\}.
}
$$

As a percentage, multiply by 100.

For example, 72 correct predictions give

$$
\frac{72}{76}\times100\%
=94.7368\ldots\%
\approx94.74\%.
$$

## 28. Evaluation: Top-5 accuracy

Top-5 asks whether the true identity appears anywhere among the five highest-ranked identities.

Let $\operatorname{Top5}(i)$ denote the set of the five highest-ranked identities for query $i$. Then

$$
\boxed{
\operatorname{Top5Accuracy}
=\frac{1}{76}\sum_{i=1}^{76}
\mathbf 1\{c_i\in\operatorname{Top5}(i)\}.
}
$$

A query can be wrong at Rank 1 but correct at Top-5. For example, if the true identity is $s_{19}$ and the ranking begins

$$
s_8,\ s_{19},\ s_{11},\ s_{32},\ s_1,
$$

then Rank-1 is wrong but Top-5 is correct.

For the same ranking, Top-5 accuracy cannot be smaller than Rank-1 accuracy.

## 29. Results reported in the project

| Method | Rank-1 correct | Rank-1 wrong | Rank-1 accuracy | Top-5 accuracy |
|---|---:|---:|---:|---:|
| Eigenfaces | 72 | 4 | 94.74% | 97.37% |
| Fisherfaces | 74 | 2 | 97.37% | 100.00% |

Fisherfaces makes two fewer Rank-1 errors on this test split. Its Rank-1 accuracy is higher by approximately

$$
97.37\%-94.74\%=\boxed{2.63\text{ percentage points}}.
$$

Fisherfaces Top-5 accuracy is 100%, meaning the correct identity appears among the five candidates for all 76 test images. It does **not** mean that every Rank-1 prediction is correct.

## 30. Exact Rank-1 errors recorded in the report

| Test image number | True identity | Eigenfaces prediction | Fisherfaces prediction |
|---:|---|---|---|
| 2 | $s_1$ | $s_{17}$ — wrong | $s_{17}$ — wrong |
| 10 | $s_5$ | $s_{36}$ — wrong | $s_5$ — correct |
| 20 | $s_{10}$ | $s_8$ — wrong | $s_{10}$ — correct |
| 37 | $s_{19}$ | $s_8$ — wrong | $s_8$ — wrong |

The two errors shared by both methods are test images 2 and 37. Fisherfaces corrects the additional Eigenfaces errors on test images 10 and 20.

The report notes that the true identity is still in Fisherfaces' Top-5 list for the two shared errors:
- For test image 2, the true identity $s_1$ is fourth.
- For test image 37, the true identity $s_{19}$ is second.

## 31. Why might Fisherfaces perform better in this experiment?

The explanation follows from the different mathematical objectives.

### Eigenfaces

PCA preserves directions of large overall variation. A direction can have large variation without being especially useful for distinguishing identities.

### Fisherfaces

LDA uses identity labels and seeks directions with large between-identity scatter relative to within-identity scatter.

Therefore, the resulting 37-dimensional Fisherface representation may make the known identities easier to separate in this dataset.

This is an interpretation of the observed result, **not a theorem that Fisherfaces must always outperform Eigenfaces**. Another dataset, train/test split, lighting condition, or parameter choice could produce a different result.

Fisherfaces also has fewer final coordinates (37 rather than 155), which can reduce the cost of feature comparisons. However, the total training cost also includes PCA and the generalized eigenvalue calculation.

## 32. Complete mathematical pipeline for one query

Let $x$ be a new preprocessed test face, represented as a vector in $\mathbb R^{4800}$.

### Eigenfaces route

1. Center the face:

$$
x_c=x-\mu.
$$

2. Project onto the 155 selected eigenfaces:

$$
z_E=U_{\text{EIG}}^Tx_c\in\mathbb R^{155}.
$$

3. Normalize:

$$
f_E(x)=\frac{z_E}{\|z_E\|_2}
$$

(with the implementation's $10^{-8}$ denominator floor).

4. Compare with each normalized Eigenfaces gallery vector using a dot product.

### Fisherfaces route

1. Center the face:

$$
x_c=x-\mu.
$$

2. Project onto the 37 Fisherface directions:

$$
z_F=W_{\text{FISH}}^Tx_c\in\mathbb R^{37}.
$$

3. Normalize:

$$
f_F(x)=\frac{z_F}{\|z_F\|_2}.
$$

4. Compare with each normalized Fisherfaces gallery vector using a dot product.

### Shared identity decision

For either method $M\in\{E,F\}$,

$$
S_c^{(M)}(x)=
\max_{r=1,\ldots,8}
f_M(x)^T G_{c,r}^{(M)}.
$$

Then

$$
\boxed{
\widehat c_M(x)=
\arg\max_{c\in\{s_1,\ldots,s_{38}\}}
S_c^{(M)}(x).
}
$$

The two methods differ in their learned feature directions. Their gallery matching, maximum-template rule, cosine similarity, and ranking logic are the same.

## 33. Complete system flow

The two learning routes share the same preprocessing and final matching system.

### Common input preparation

$$
\text{Image}
\rightarrow\text{orientation correction}
\rightarrow\text{grayscale}
\rightarrow 60\times80\text{ resize}
\rightarrow\text{histogram equalization}
\rightarrow\text{flatten}
\rightarrow\text{pixel standardization}.
$$

Every image becomes a 4,800-dimensional vector.

### Route A: Eigenfaces

$$
X_G
\rightarrow\text{training mean and centering}
\rightarrow\text{Gram matrix PCA}
\rightarrow\text{155 selected eigenfaces}
\rightarrow\text{normalized features}.
$$

### Route B: Fisherfaces

$$
X_G
\rightarrow\text{training mean and centering}
\rightarrow\text{PCA to 300 dimensions}
\rightarrow S_W,S_B
\rightarrow\text{regularized LDA}
\rightarrow\text{37 Fisherfaces}
\rightarrow\text{normalized features}.
$$

### Shared recognition stage

$$
\text{query/gallery dot products}
\rightarrow\text{cosine similarities}
\rightarrow\text{maximum score per identity}
\rightarrow\text{rank 38 identities}
\rightarrow\text{Rank-1 and Top-5 evaluation}.
$$

## 34. Implementation settings and dimensions to remember

| Setting / quantity | Value | Explanation |
|---|---:|---|
| `FACE_SIZE` | $(60,80)$ | Width 60, height 80 |
| Pixels per image | 4,800 | $60\times80$ |
| `EIGEN_VAR` | 0.95 | PCA cumulative variance target |
| `EIGEN_MAX` | 300 | Maximum Eigenfaces selection boundary |
| `EIGEN_DROP_FIRST` | 3 | First three PCA directions omitted |
| Eigenfaces boundary | 158 | First 158 explain 95.06% |
| Eigenfaces final dimension | 155 | Components 4–158 |
| `FISHER_PCA` | 300 | PCA dimension before LDA |
| `FISHER_DIM` | 200 | Upper cap, not final dimension |
| `FISHER_RIDGE` | $10^{-2}$ | Ridge scale |
| Fisherfaces final dimension | 37 | $38-1=37$ |
| Gallery / test count | 304 / 76 | Eight train and two test per identity |

The five largest Fisher criteria printed in the saved run are

$$
982.933,\quad840.408,\quad721.602,\quad412.236,\quad382.488.
$$

These are generalized eigenvalues/Fisher criteria under the scatter-matrix scaling. They are not probabilities or accuracy percentages.

## 35. Important implementation caveats

1. **Closed-set only:** The system always chooses one of $s_1,\ldots,s_{38}$. There is no unknown-person rejection threshold.
2. **Similarity is not probability:** A cosine score of 0.80 does not mean an 80% chance that the prediction is correct.
3. **Fixed split:** There are 76 test images, only two per identity, and one predefined split. The percentages describe this experiment, not guaranteed future performance.
4. **File checks are not a full quality audit:** The notebook checks that expected folders and files exist. It does not implement duplicate detection, blur filtering, occlusion checks, face detection, landmark alignment, or label verification.
5. **No real-time recognition:** The project does not implement live camera tracking or real-time recognition.
6. **No deep neural network:** The implementation uses classical linear algebra with NumPy, SciPy, Matplotlib, and Pillow. It does not use a pretrained deep face embedding model.
7. **No full exported prediction file:** The saved run prints predictions and displays figures; it does not save a CSV or spreadsheet of predictions.
8. **New-image preprocessing must match:** A new image must go through the same orientation, grayscale, resize, equalization, flattening, scaling, and standardization pipeline. The training mean and projection matrices must be reused.

## 36. Glossary

| Term | Meaning |
|---|---|
| Face vector | A 4,800-entry standardized grayscale image vector |
| Training matrix | $4800\times304$ matrix of training faces |
| Test matrix | $4800\times76$ matrix of test faces |
| Gallery | Stored training feature vectors with identity labels |
| Query | A test image to be identified |
| Mean face | Pixel-by-pixel average of the training faces |
| Centering | Subtracting the training mean face |
| PCA | Finds directions of maximum overall variation |
| Eigenvalue | Measures variation associated with a PCA direction |
| Eigenvector | A direction associated with an eigenvalue |
| Eigenface | A PCA direction in pixel space |
| Gram matrix | $X_c^TX_c$, the smaller sample-space PCA matrix |
| Explained variance | Fraction of total variation captured by selected PCA components |
| LDA | Finds directions that separate known classes |
| Within-class scatter | Variation among examples of the same identity |
| Between-class scatter | Separation among identity means |
| Regularization | Adding a diagonal term to stabilize a matrix |
| Fisherface | A PCA-plus-LDA discriminant direction mapped to pixel space |
| L2 norm | Euclidean length of a vector |
| Cosine similarity | Similarity based on the angle between vectors |
| Identity-level score | Maximum similarity across one identity's eight templates |
| Rank-1 | Identity with the highest score |
| Top-5 | Five highest-ranked identities |
| Closed-set | Every query is assumed to belong to a known identity |